In [1]:
import os
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv  
from langchain_core.messages import BaseMessage # The foundational class for all message types in LangGraph
from langchain_core.messages import SystemMessage # Message for providing instructions to the LLM
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode


load_dotenv()

google_api_key = os.environ["GOOGLE_API_KEY"]

free_model_names = [
    # Current stable Gemini models
    "gemini-3.8-flash",
    "gemini-3.7-flash",
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",

    # Preview models
    "gemini-3-flash-preview",

    # Legacy models - restricted/deprecated
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.5-pro",
]

models = [
    ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=google_api_key,
        max_retries=0,
        timeout=30,
    )
    for model_name in free_model_names
]
llm = models[0].with_fallbacks(models[1:])

In [3]:
from langchain_core.tools import tool

@tool
def get_weather(location: str) -> str:
    "get the weather of the location"
    return f"Processed: {location}"

model_with_tools = llm.bind_tools([get_weather])

In [4]:
print(model_with_tools.invoke("New York"))

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_16099777548849282477': 'EqgDCqUDAWkUfRP/deYfpHVOyi0dqO/+nzkPs0zppn/EWGsmS7GgpfQXQVlKu90WZUS9ff90N5C2o5QoUK6L8wY8v2mbCTSYyaHCyQn2gmyCtWfZgiTUQ6fcA8jrk7Jh5gcGSxYC+ql8600l0eGXxgcQ97wrz4jrCvJhKthpXW+5aKPbqXqRbDpxKN1WXWJ0LLxj647b51dsI4uoYwWP5xhV6zMsqhEeNEgicnifKwpBWANX3Rg3+MHRksFxCYQArgpJONvCE+W1Vtk/ab4L4+P+sFwqq1RuafajnLb1+/041dus/1UtcrQpj7Hs0wMq0A59oUcILxtBHfdGXLq2SeBcWohT3+i9zncsPKSGrGbdqyueaZMPLw2LYsXoOFiv7FSF/N6D5lDKN9hWnXvld5mAvUFJosAVGieWwcqQazfBq+KUqhNJh98VAxSDEjerVhEZZx4JhdYT8BzxeG1jptwhSw2FRbELYuPeKz65bfJdHbq8AkHEir+t642+1rqouuRpH1AuzSl+CO5uCeQ+dMQTRNT+40GmPtvfVum0Qqb/rlP/bPiI3f32ug=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1e019-ddf5-7774-8d14-2104e60f35c1-0' tool_calls=[{'name': 'get_weather', 'args': {'l

In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather service processed your request for Boston, but did not return detailed conditions. Please try again or specify more details if needed.
